# Top and Bottom of the Premiere League

BIn 1992/93, Manchester United won the first Premier League title with 84 points.
Coventry City survived relegation with 49. The gap: **35 points**.

In 2019/20, Liverpool won with 99 points. Aston Villa survived with 34.
The gap: **65 points**.

Is the Premier League becoming a two-tier league? And if so, when did the
divergence accelerate? This analysis pulls live data from Wikipedia to find out.

## Data Collection

The dataset is scraped live from Wikipedia's season-by-season Premier League
articles. For each of the 33 completed seasons, the scraper extracts:

- **Champion** and their points total
- **Top 4 finishers** (Champions League qualifiers since 2001/02)
- **17th-place finisher** (the lowest team to survive relegation)
- **Relegated teams** (18th, 19th, 20th)

In [1]:
# Run once
#!pip install plotly pandas numpy html5lib

In [2]:
#| label: scraper
#| code-summary: "Web scraping code"

import pandas as pd
import numpy as np
import requests
import time
import re
import warnings
from pathlib import Path
import io
import html5lib

warnings.filterwarnings("ignore", message=".*lxml.*")

In [3]:
def get_season_url(start_year: int) -> str:
   """Generate the Wikipedia URL for a Premier League season article."""
   end_short = str(start_year + 1)[-2:]
   # Wikipedia uses en-dash (\u2013), not hyphen
   season = f"{start_year}\u201399" if end_short == "99" else f"{start_year}\u2013{end_short}"

   # The league dropped "FA" from its name in 2007/08
   if start_year <= 2006:
       return f"https://en.wikipedia.org/wiki/{season}_FA_Premier_League"
   else:
       return f"https://en.wikipedia.org/wiki/{season}_Premier_League"


def clean_team_name(name: str) -> str:
   """Remove Wikipedia footnote markers and qualification tags from team names."""
   if not isinstance(name, str):
       return str(name)
   # Remove content in parentheses like (C), (R), (Q)
   name = re.sub(r"\s*\([A-Za-z, ]+\)", "", name)
   # Remove bracketed citation numbers like [a], [1], [note 1]
   name = re.sub(r"\s*\[.*?\]", "", name)
   # Remove trailing special characters
   name = name.strip().rstrip("*").strip()
   return name


def find_league_table(tables: list, num_teams: int) -> pd.DataFrame | None:
   """Identify the league standings table from a list of parsed HTML tables."""
   for table in tables:
       # Flatten MultiIndex columns if present
       if isinstance(table.columns, pd.MultiIndex):
           table.columns = [
               str(c[-1]) if isinstance(c, tuple) else str(c)
               for c in table.columns
           ]

       col_str = " ".join(str(c) for c in table.columns).lower()

       # Look for a table with the right row count and typical column names
       has_pts = "pts" in col_str or "points" in col_str
       has_team = "team" in col_str or "club" in col_str
       right_size = len(table) == num_teams

       if has_pts and right_size:
           return table.copy()

   # Fallback: any table with num_teams rows containing numeric-looking data
   for table in tables:
       if len(table) == num_teams and table.shape[1] >= 8:
           return table.copy()

   return None


def scrape_season(start_year: int) -> dict | None:
   """
   Scrape a single Premier League season from Wikipedia.

   Returns a dict with champion, top 4, 17th place (survival), and relegated teams.
   """
   url = get_season_url(start_year)
   num_teams = 22 if start_year <= 1994 else 20

   try:
       resp = requests.get(
           url,
           headers={"User-Agent": "PL-Portfolio-Project/1.0 (educational)"},
           timeout=15,
       )
       resp.raise_for_status()
   except requests.RequestException as e:
       print(f"  ERROR fetching {start_year}/{start_year+1}: {e}")
       return None

   try:
       tables = pd.read_html(io.StringIO(resp.text), flavor='html5lib')
   except ValueError:
       print(f"  ERROR parsing tables for {start_year}/{start_year+1}")
       return None

   table = find_league_table(tables, num_teams)
   if table is None:
       print(f"  WARNING: Could not find league table for {start_year}/{start_year+1}")
       return None

   # Normalize column names
   table.columns = [str(c).strip() for c in table.columns]

   # Identify key columns
   pts_col = next((c for c in table.columns if c.lower() in ("pts", "points")), None)
   team_col = next(
       (c for c in table.columns if c.lower() in ("team", "club")), None
   )

   if pts_col is None or team_col is None:
       # Try positional fallback: team is usually col 1, pts is last
       team_col = table.columns[1]
       pts_col = table.columns[-1]

   # Clean data
   table[team_col] = table[team_col].apply(clean_team_name)
   table[pts_col] = pd.to_numeric(table[pts_col], errors="coerce")

   # Sort by points descending (table should already be sorted, but just in case)
   table = table.sort_values(pts_col, ascending=False).reset_index(drop=True)

   # Determine relegation and survival positions
   if num_teams == 22:
       survival_pos = 18  # 19th place (0-indexed 18) survived; 20-22 relegated
       relegated_start = 19
   else:
       survival_pos = 16  # 17th place (0-indexed 16) survived; 18-20 relegated
       relegated_start = 17

   end_short = str(start_year + 1)[-2:]
   season_label = f"{start_year}/{end_short}"

   return {
       "Season": season_label,
       "Champion": table.iloc[0][team_col],
       "Title-Winning Points": int(table.iloc[0][pts_col]),
       "2nd Place": table.iloc[1][team_col],
       "2nd Place Points": int(table.iloc[1][pts_col]),
       "3rd Place": table.iloc[2][team_col],
       "3rd Place Points": int(table.iloc[2][pts_col]),
       "4th Place": table.iloc[3][team_col],
       "4th Place Points": int(table.iloc[3][pts_col]),
       "Survived Relegation": table.iloc[survival_pos][team_col],
       "Relegation Survival Points": int(table.iloc[survival_pos][pts_col]),
       "Relegated 1": table.iloc[relegated_start][team_col],
       "Relegated 2": table.iloc[relegated_start + 1][team_col],
       "Relegated 3": table.iloc[relegated_start + 2][team_col],
   }


def scrape_all_seasons(
   start: int = 1992, end: int = 2024, cache_path: str = "pl_data_cache.csv"
) -> pd.DataFrame:
   """
   Scrape all Premier League seasons, with local CSV caching.

   Uses cached data if available and not stale (< 7 days old).
   """
   cache_file = Path(cache_path)

   # Use cache if it exists and is less than 7 days old
   if cache_file.exists():
       age_days = (time.time() - cache_file.stat().st_mtime) / 86400
       if age_days < 7:
           print(f"Using cached data ({age_days:.1f} days old)")
           return pd.read_csv(cache_file)
       else:
           print(f"Cache is {age_days:.0f} days old, refreshing...")

   # Scrape each season
   seasons = []
   for year in range(start, end + 1):
       print(f"  Scraping {year}/{year+1}...")
       result = scrape_season(year)
       if result:
           seasons.append(result)
       # Be polite to Wikipedia
       time.sleep(1.5)

   df = pd.DataFrame(seasons)

   # Save to cache
   df.to_csv(cache_file, index=False)
   print(f"\nDone. {len(df)} seasons scraped and cached to {cache_path}")

   return df

In [4]:
#| label: load-data
#| code-summary: "Load data (from cache or scrape)"

df = scrape_all_seasons(start=1992, end=2024)

# Computed columns
df["Gap"] = df["Title-Winning Points"] - df["Relegation Survival Points"]
df["Ratio"] = (df["Title-Winning Points"] / df["Relegation Survival Points"]).round(2)

print(f"Dataset: {len(df)} seasons loaded")
df[["Season", "Champion", "Title-Winning Points", "Survived Relegation",
   "Relegation Survival Points", "Relegated 1", "Relegated 2", "Relegated 3"]].tail(10)

Using cached data (0.0 days old)
Dataset: 33 seasons loaded


,Season,Champion,Title-Winning Points,Survived Relegation,Relegation Survival Points,Relegated 1,Relegated 2,Relegated 3
23,2015/16,Leicester City,81,Sunderland,39,Newcastle United,Norwich City,Aston Villa
24,2016/17,Chelsea,93,Watford,40,Hull City,Middlesbrough,Sunderland
25,2017/18,Manchester City,100,Southampton,36,Swansea City,Stoke City,West Bromwich Albion
26,2018/19,Manchester City,98,Brighton & Hove Albion,36,Cardiff City,Fulham,Huddersfield Town
27,2019/20,Liverpool,99,Aston Villa,35,Bournemouth,Watford,Norwich City
28,2020/21,Manchester City,86,Burnley,39,Fulham,West Bromwich Albion,Sheffield United
29,2021/22,Manchester City,93,Leeds United,38,Burnley,Watford,Norwich City
30,2022/23,Manchester City,89,Everton,36,Leicester City,Leeds United,Southampton
31,2023/24,Manchester City,91,Burnley,24,Sheffield United,Everton,Nottingham Forest
32,2024/25,Liverpool,84,Tottenham Hotspur,38,Leicester City,Ipswich Town,Southampton


## Champions League & Relegation Overview

Before diving into the trend analysis, here's a quick look at which clubs have
dominated the top 4 and which have faced the drop.

In [5]:
#| label: tbl-top4-frequency
#| tbl-cap: "Most frequent Top 4 finishers (1992-2025)"
#| code-summary: "Top 4 frequency analysis"

# Count top-4 appearances for each club
top4_clubs = pd.concat([
   df["Champion"],
   df["2nd Place"],
   df["3rd Place"],
   df["4th Place"],
]).value_counts().reset_index()
top4_clubs.columns = ["Club", "Top 4 Finishes"]
top4_clubs["Percentage"] = (top4_clubs["Top 4 Finishes"] / len(df) * 100).round(1)

top4_clubs.head(10).style.format({"Percentage": "{:.1f}%"}).hide(axis="index")

Club,Top 4 Finishes,Percentage
Manchester United,26,78.8%
Arsenal,24,72.7%
Liverpool,21,63.6%
Chelsea,20,60.6%
Manchester City,15,45.5%
Tottenham Hotspur,7,21.2%
Newcastle United,6,18.2%
Blackburn Rovers,3,9.1%
Aston Villa,3,9.1%
Leeds United,3,9.1%


In [6]:
#| label: tbl-relegated
#| tbl-cap: "Most frequently relegated clubs (1992-2025)"
#| code-summary: "Relegation frequency analysis"

# Count relegations for each club
relegated_clubs = pd.concat([
   df["Relegated 1"],
   df["Relegated 2"],
   df["Relegated 3"],
]).value_counts().reset_index()
relegated_clubs.columns = ["Club", "Times Relegated"]

relegated_clubs.head(10).style.hide(axis="index")

Club,Times Relegated
Norwich City,6
Leicester City,5
West Bromwich Albion,5
Sheffield United,4
Sunderland,4
Middlesbrough,4
Nottingham Forest,4
Watford,4
Crystal Palace,3
Bolton Wanderers,3


## Interactive Analysis

Use the **dropdown menu** in the top-left of the chart to explore different eras.
Each era recalculates its own trend line independently.

Hover over any point to see the champion, the surviving team, or the full top 4.


In [7]:
#| label: fig-main
#| fig-cap: "Premier League points analysis across three dimensions: absolute points, gap, and ratio. Use the dropdown to switch between eras."
#| code-summary: "Main visualization code"

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
   rows=3, cols=1,
   shared_xaxes=True,
   vertical_spacing=0.08,
   row_heights=[0.50, 0.25, 0.25],
   subplot_titles=(
       "Title-Winning vs. Relegation Survival Points",
       "Points Gap (Title Winner minus Survival)",
       "Points Ratio (Title Winner / Survival)",
   ),
)

# ─── Era definitions ───

eras = [
   {"label": "Full History (1992-2025)", "start": "1992/93", "end": "2024/25",
    "title": "Premier League: The Growing Divide (1992-2025)"},
   {"label": "Pre-Abramovich (1992-2003)", "start": "1992/93", "end": "2002/03",
    "title": "Premier League: Pre-Abramovich Era (1992-2003)"},
   {"label": "Oligarch Era (2003-2016)", "start": "2003/04", "end": "2015/16",
    "title": "Premier League: The Oligarch Era (2003-2016)"},
   {"label": "Pep/Klopp Era (2016-2025)", "start": "2016/17", "end": "2024/25",
    "title": "Premier League: The Pep/Klopp Era (2016-2025)"},
   {"label": "Post-COVID (2020-2025)", "start": "2020/21", "end": "2024/25",
    "title": "Premier League: Post-COVID Era (2020-2025)"},
]

TRACES_PER_ERA = 9

outlier_config = [
   {"season": "2003/04", "y": 90, "text": "Invincibles",
    "color": "#FFD54F", "ax": -50, "ay": -30},
   {"season": "2015/16", "y": 81, "text": "Leicester!",
    "color": "#66FF66", "ax": -50, "ay": -30},
   {"season": "2017/18", "y": 100, "text": "100 pts",
    "color": "#4FC3F7", "ax": 0, "ay": -25},
   {"season": "2019/20", "y": 99, "text": "COVID",
    "color": "#EF5350", "ax": 40, "ay": -20},
]

era_annotations = {}

# ─── Build 9 traces per era ───

for i, era in enumerate(eras):
   start_idx = df[df["Season"] == era["start"]].index[0]
   end_idx = df[df["Season"] == era["end"]].index[0]
   era_df = df.iloc[start_idx:end_idx + 1].reset_index(drop=True)
   x_num = np.arange(len(era_df))

   # Build rich hover text for title-winning points
   title_hover = [
       f"<b>{row['Season']}</b><br>"
       f"\U0001F3C6 {row['Champion']}: {row['Title-Winning Points']} pts<br>"
       f"\U0001F948 {row['2nd Place']}: {row['2nd Place Points']} pts<br>"
       f"\U0001F949 {row['3rd Place']}: {row['3rd Place Points']} pts<br>"
       f"4th: {row['4th Place']}: {row['4th Place Points']} pts"
       for _, row in era_df.iterrows()
   ]

   # Build rich hover text for relegation survival
   releg_hover = [
       f"<b>{row['Season']}</b><br>"
       f"\U0001FA82 Survived: {row['Survived Relegation']} ({row['Relegation Survival Points']} pts)<br>"
       f"\u274C {row['Relegated 1']}<br>"
       f"\u274C {row['Relegated 2']}<br>"
       f"\u274C {row['Relegated 3']}"
       for _, row in era_df.iterrows()
   ]

   # Trace 0: Title-Winning Points
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=era_df["Title-Winning Points"],
       name="Title-Winning Points", mode="lines+markers",
       line=dict(color="#4FC3F7", width=3), marker=dict(size=7),
       hovertext=title_hover, hoverinfo="text",
       visible=False, showlegend=True,
   ), row=1, col=1)

   # Trace 1: Title Trend
   t_c = np.polyfit(x_num, era_df["Title-Winning Points"], 1)
   t_t = np.polyval(t_c, x_num)
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=t_t,
       name=f"Title Trend ({t_c[0]:+.2f} pts/yr)", mode="lines",
       line=dict(color="#4FC3F7", width=2, dash="dash"),
       hoverinfo="skip", visible=False, showlegend=True,
   ), row=1, col=1)

   # Trace 2: Relegation Survival Points
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=era_df["Relegation Survival Points"],
       name="Relegation Survival Points", mode="lines+markers",
       line=dict(color="#EF5350", width=3), marker=dict(size=7),
       hovertext=releg_hover, hoverinfo="text",
       visible=False, showlegend=True,
   ), row=1, col=1)

   # Trace 3: Relegation Trend
   r_c = np.polyfit(x_num, era_df["Relegation Survival Points"], 1)
   r_t = np.polyval(r_c, x_num)
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=r_t,
       name=f"Relegation Trend ({r_c[0]:+.2f} pts/yr)", mode="lines",
       line=dict(color="#EF5350", width=2, dash="dash"),
       hoverinfo="skip", visible=False, showlegend=True,
   ), row=1, col=1)

   # Trace 4: Shaded gap
   fig.add_trace(go.Scatter(
       x=era_df["Season"].tolist() + era_df["Season"].tolist()[::-1],
       y=era_df["Title-Winning Points"].tolist()
         + era_df["Relegation Survival Points"].tolist()[::-1],
       fill="toself", fillcolor="rgba(255, 255, 255, 0.05)",
       line=dict(width=0), showlegend=False, hoverinfo="skip", visible=False,
   ), row=1, col=1)

   # Trace 5: Gap Bars
   gap_colors = [
       f"rgba({min(255, int(150 + (g - 30) * 3))}, "
       f"{max(80, int(220 - (g - 30) * 4))}, 100, 0.85)"
       for g in era_df["Gap"]
   ]
   fig.add_trace(go.Bar(
       x=era_df["Season"], y=era_df["Gap"], name="Points Gap",
       marker=dict(color=gap_colors, line=dict(width=0)),
       hovertemplate="<b>%{x}</b><br>Gap: %{y} pts<extra></extra>",
       visible=False, showlegend=True,
   ), row=2, col=1)

   # Trace 6: Gap Trend
   g_c = np.polyfit(x_num, era_df["Gap"], 1)
   g_t = np.polyval(g_c, x_num)
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=g_t,
       name=f"Gap Trend ({g_c[0]:+.2f} pts/yr)", mode="lines",
       line=dict(color="#FFD54F", width=2, dash="dash"),
       hoverinfo="skip", visible=False, showlegend=True,
   ), row=2, col=1)

   # Trace 7: Ratio Line
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=era_df["Ratio"], name="Points Ratio",
       mode="lines+markers", line=dict(color="#AB47BC", width=3),
       marker=dict(size=6),
       hovertemplate="<b>%{x}</b><br>Ratio: %{y:.2f}x<extra></extra>",
       visible=False, showlegend=True,
   ), row=3, col=1)

   # Trace 8: Ratio Trend
   rt_c = np.polyfit(x_num, era_df["Ratio"], 1)
   rt_t = np.polyval(rt_c, x_num)
   fig.add_trace(go.Scatter(
       x=era_df["Season"], y=rt_t,
       name=f"Ratio Trend ({rt_c[0]:+.3f}/yr)", mode="lines",
       line=dict(color="#AB47BC", width=2, dash="dash"),
       hoverinfo="skip", visible=False, showlegend=True,
   ), row=3, col=1)

   # Annotations for this era
   ann_for_era = []
   for outlier in outlier_config:
       if outlier["season"] in era_df["Season"].values:
           ann_for_era.append(dict(
               x=outlier["season"], y=outlier["y"], text=outlier["text"],
               showarrow=True, arrowhead=2, ax=outlier["ax"], ay=outlier["ay"],
               font=dict(color=outlier["color"], size=10, family="Arial Black"),
               xref="x", yref="y",
           ))
   era_annotations[i] = ann_for_era

# Set first era visible
for j in range(TRACES_PER_ERA):
   fig.data[j].visible = True

# 2.5x reference line on ratio subplot
fig.add_shape(
   type="line", x0=0, x1=1, y0=2.5, y1=2.5,
   xref="x3 domain", yref="y3",
   line=dict(dash="dot", color="rgba(255,255,255,0.3)"),
)

hline_label = dict(
   x=1.01, y=2.5, xref="x3 domain", yref="y3",
   text="2.5x", showarrow=False,
   font=dict(color="rgba(255,255,255,0.5)", size=11), xanchor="left",
)
for i in era_annotations:
   era_annotations[i].append(hline_label)

# Dropdown buttons
total_traces = TRACES_PER_ERA * len(eras)
dropdown_buttons = []
for i, era in enumerate(eras):
   vis = [False] * total_traces
   for j in range(TRACES_PER_ERA):
       vis[i * TRACES_PER_ERA + j] = True
   dropdown_buttons.append(dict(
       label=era["label"], method="update",
       args=[{"visible": vis},
             {"title.text": era["title"], "annotations": era_annotations[i]}],
   ))

# Layout
fig.update_layout(
   template="plotly_dark",
   title=dict(text=eras[0]["title"], font=dict(size=20)),
   height=1000, width=1100, hovermode="closest",
   legend=dict(orientation="h", yanchor="bottom", y=1.05,
               xanchor="center", x=0.5, font=dict(size=10)),
   margin=dict(t=150),
   annotations=era_annotations[0],
   updatemenus=[dict(
       type="dropdown", direction="down", active=0,
       x=0.0, xanchor="left", y=1.18, yanchor="top",
       bgcolor="rgba(50, 50, 50, 0.9)",
       bordercolor="rgba(255, 255, 255, 0.3)",
       font=dict(color="white", size=12),
       buttons=dropdown_buttons,
   )],
)

fig.update_yaxes(title_text="Points", range=[20, 108], row=1, col=1)
fig.update_yaxes(title_text="Gap (pts)", row=2, col=1)
fig.update_yaxes(title_text="Ratio", row=3, col=1)
fig.update_xaxes(tickangle=-45, dtick=2, row=3, col=1)

fig.show()

## The Top 4 Lock-Out

One of the most striking patterns is how few clubs have occupied the Champions
League places over 33 seasons.

In [8]:
#| label: fig-top4-pie
#| fig-cap: "Distribution of Top 4 finishes across all 33 Premier League seasons (132 total slots)"
#| code-summary: "Top 4 concentration chart"

top4_for_chart = top4_clubs.head(8).copy()
others_count = top4_clubs.iloc[8:]["Top 4 Finishes"].sum()
top4_for_chart = pd.concat([
   top4_for_chart,
   pd.DataFrame([{"Club": "All Others", "Top 4 Finishes": others_count, "Percentage": 0}])
], ignore_index=True)

fig_pie = go.Figure(data=[go.Pie(
   labels=top4_for_chart["Club"],
   values=top4_for_chart["Top 4 Finishes"],
   hole=0.4,
   textinfo="label+percent",
   textposition="outside",
   marker=dict(colors=[
       "#4FC3F7", "#EF5350", "#66BB6A", "#AB47BC",
       "#FFD54F", "#FF7043", "#26C6DA", "#8D6E63", "#BDBDBD"
   ]),
   hovertemplate="<b>%{label}</b><br>Top 4 finishes: %{value}<br>%{percent}<extra></extra>",
)])

fig_pie.update_layout(
   template="plotly_dark",
   title=dict(text="Top 4 Finishes: Who Owns the Champions League Places?", font=dict(size=16)),
   height=500, width=800,
   annotations=[dict(text="132<br>slots", x=0.5, y=0.5, font_size=16,
                     showarrow=False, font_color="white")],
)

fig_pie.show()

## The Relegation Revolving Door

While the top is locked down by a handful of clubs, the bottom is chaos.


In [9]:
#| label: fig-relegated-bar
#| fig-cap: "Clubs relegated most often from the Premier League (1992-2025)"
#| code-summary: "Relegation frequency chart"

releg_chart = relegated_clubs.head(12).copy()

fig_releg = go.Figure(data=[go.Bar(
   x=releg_chart["Club"],
   y=releg_chart["Times Relegated"],
   marker=dict(
       color=releg_chart["Times Relegated"],
       colorscale="Reds",
       line=dict(width=0),
   ),
   hovertemplate="<b>%{x}</b><br>Relegated %{y} times<extra></extra>",
)])

fig_releg.update_layout(
   template="plotly_dark",
   title=dict(text="The Usual Suspects: Most Relegated Premier League Clubs", font=dict(size=16)),
   xaxis=dict(title="", tickangle=-45),
   yaxis=dict(title="Times Relegated", dtick=1),
   height=450, width=800,
)

fig_releg.show()

## Season-by-Season Detail

In [10]:
#| label: tbl-full-data
#| tbl-cap: "Complete Premier League season data (scroll to see all)"
#| code-summary: "Full data table"

display_df = df[[
   "Season", "Champion", "Title-Winning Points",
   "2nd Place", "3rd Place", "4th Place",
   "Survived Relegation", "Relegation Survival Points",
   "Relegated 1", "Relegated 2", "Relegated 3", "Gap",
]].copy()

display_df.columns = [
   "Season", "Champion", "Champ Pts",
   "2nd", "3rd", "4th",
   "Survived", "Survival Pts",
   "Relegated", "Relegated", "Relegated", "Gap",
]

# Rename duplicate "Relegated" columns
display_df.columns = [
   "Season", "Champion", "Champ Pts",
   "2nd", "3rd", "4th",
   "Survived", "Survival Pts",
   "Rel. 1", "Rel. 2", "Rel. 3", "Gap",
]

display_df.style.background_gradient(
   subset=["Gap"], cmap="YlOrRd"
).hide(axis="index")

Season,Champion,Champ Pts,2nd,3rd,4th,Survived,Survival Pts,Rel. 1,Rel. 2,Rel. 3,Gap
1992/93,Manchester United,84,Aston Villa,Norwich City,Blackburn Rovers,Oldham Athletic,49,Crystal Palace,Middlesbrough,Nottingham Forest,35
1993/94,Manchester United,92,Blackburn Rovers,Newcastle United,Arsenal,Ipswich Town,43,Sheffield United,Oldham Athletic,Swindon Town,49
1994/95,Blackburn Rovers,89,Manchester United,Nottingham Forest,Liverpool,Crystal Palace,45,Norwich City,Leicester City,Ipswich Town,44
1995/96,Manchester United,82,Newcastle United,Liverpool,Aston Villa,Southampton,38,Manchester City,Queens Park Rangers,Bolton Wanderers,44
1996/97,Manchester United,75,Arsenal,Liverpool,Newcastle United,Coventry City,41,Sunderland,Nottingham Forest,Middlesbrough,34
1997/98,Arsenal,78,Manchester United,Liverpool,Chelsea,Everton,40,Bolton Wanderers,Barnsley,Crystal Palace,38
1998/99,Manchester United,79,Arsenal,Chelsea,Leeds United,Southampton,41,Charlton Athletic,Blackburn Rovers,Nottingham Forest,38
1999/00,Manchester United,91,Arsenal,Leeds United,Liverpool,Bradford City,36,Wimbledon,Sheffield Wednesday,Watford,55
2000/01,Manchester United,80,Arsenal,Liverpool,Leeds United,Derby County,42,Manchester City,Coventry City,Bradford City,38
2001/02,Arsenal,87,Liverpool,Manchester United,Newcastle United,Sunderland,40,Ipswich Town,Derby County,Leicester City,47


## Era-by-Era Breakdown

### Pre-Abramovich (1992-2003): The Fergie Monopoly

The early Premier League was dominated by a single manager. Sir Alex Ferguson's
Manchester United won 8 of 11 titles, but the points required to win were
**not exceptionally high** (average: ~83 pts). The league was competitive enough
that Blackburn (1995) and Arsenal (1998, 2002, 2004) could break through.

Relegation survival averaged **~39 points**, notably higher than any subsequent
era. The financial gap between top and bottom had not yet become a chasm.

### The Oligarch Era (2003-2016): Money Changes Everything

Roman Abramovich's purchase of Chelsea in 2003 marked a turning point. Then
Manchester City's 2008 takeover created a third "super-club." Title-winning
points climbed to an average of **~88 pts**, while relegation survival drifted
down to **~35 points**.

The Leicester miracle (2015/16, 81 pts) was the exception that proved the rule.

### The Pep/Klopp Era (2016-2025): The Arms Race


In [11]:
#| label: tbl-era-comparison
#| tbl-cap: "Points comparison across Premier League eras"
#| code-summary: "Era comparison table"

era_stats = []
for era in eras[1:]:
   start_idx = df[df["Season"] == era["start"]].index[0]
   end_idx = df[df["Season"] == era["end"]].index[0]
   era_slice = df.iloc[start_idx:end_idx + 1]
   era_stats.append({
       "Era": era["label"].split("(")[0].strip(),
       "Seasons": len(era_slice),
       "Avg Title Pts": round(era_slice["Title-Winning Points"].mean(), 1),
       "Avg Survival Pts": round(era_slice["Relegation Survival Points"].mean(), 1),
       "Avg Gap": round(era_slice["Gap"].mean(), 1),
       "Avg Ratio": round(era_slice["Ratio"].mean(), 2),
   })

era_comparison = pd.DataFrame(era_stats)
era_comparison.style.format({"Avg Ratio": "{:.2f}x"}).hide(axis="index")

Era,Seasons,Avg Title Pts,Avg Survival Pts,Avg Gap,Avg Ratio
Pre-Abramovich,11,83.600000,41.700000,41.900000,2.02x
Oligarch Era,13,87.700000,37.200000,50.500000,2.36x
Pep/Klopp Era,9,92.600000,35.800000,56.800000,2.64x
Post-COVID,5,88.600000,35.000000,53.600000,2.63x


The arrival of Guardiola and peak Klopp created an unprecedented arms race.
City's 100-point season (2017/18) forced Liverpool to respond with 97 points
just to finish *second*. The average title total: **~92 pts**. The average
survival threshold: **~35 pts**. The gap: **~57 pts**.

## Methodology & Technical Notes

::: {.callout-note collapse="true"}
## Technical details (click to expand)

**Data source**: Scraped from Wikipedia's individual season articles using
`requests` and `pandas.read_html()`. Team names are cleaned of citation markers,
qualification tags, and Unicode artifacts.

**Caching**: Results are cached to a local CSV (`pl_data_cache.csv`) with a
7-day expiry to avoid unnecessary requests to Wikipedia. The `freeze: auto`
setting in Quarto further ensures the notebook only re-executes when source
code changes.

**Rate limiting**: A 1.5-second delay between requests respects Wikipedia's
usage guidelines.

**22-team seasons**: The first three seasons (1992/93 through 1994/95) had 22
teams. Relegation positions are adjusted accordingly (19th survived, 20th-22nd
relegated).

**Champions League qualification**: The number of CL spots changed over time
(1 in the early 1990s, gradually expanding to 4 by 2001/02). This analysis
consistently tracks the top 4 finishers regardless of how many actually
qualified for the Champions League in that era.

**Trend lines**: Linear regression via `numpy.polyfit(x, y, 1)`, calculated
independently per era.
:::

## What I'd Explore Next

- **Revenue correlation**: Does the gap track with TV deal sizes? The 2016/17
 season (start of the current mega-deal) aligns with the gap widening.
- **Wage bill ratio**: How closely does the points gap mirror the spending gap?
- **Promotion bounce-back rate**: Of the clubs relegated, how quickly (if ever)
 do they return?
- **European comparison**: Do La Liga, Bundesliga, and Serie A show the same
 stratification, or is this a Premier League-specific phenomenon?

## Source Code

The complete source for this project is available on
[GitHub](https://github.com/yourusername/premier-league-analysis).

---

*Data scraped from [Wikipedia](https://en.wikipedia.org/wiki/Premier_League).
Built with Python, Pandas, NumPy, Plotly, and Requests. Published with Quarto.*